# Micro-Expression Spotting & Classification (SAMM)

In [1]:
import os
import sys

def _find_project_root(marker="pyproject.toml", max_up=8):
    path = os.path.abspath(os.getcwd())
    for _ in range(max_up):
        if os.path.exists(os.path.join(path, marker)):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            break
        path = parent
    return os.path.abspath(os.getcwd())

project_root = _find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import glob
import cv2
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

from article.src import MELabel, FrameSequence, MESpotting, RealTimeProfiler
from src.face.modules import FaceLandmark, FaceRoiPoints
from src.face.modules.face_aligner import FaceAligner
from src.optical_flow.modules import TVL1
from src.dataset.modules.behavioral_features import BehavioralFeatures
from src.models.modules.spatio_temporal.spatio_temporal_cnn import SpatioTemporalCNN

## 1. Load SAMM annotations & initialize utilities

In [2]:
samm_dir = '/home/inadio/datasets/secondaries/samm'
annotations_path = os.path.join(samm_dir, 'annotations.xlsx')

df = pd.read_excel(annotations_path)
print(f"Loaded SAMM metadata with {len(df)} total entries.")

FPS = 200
spotter = MESpotting(cutoff_ratio=0.30, fps=FPS)
extractor = BehavioralFeatures()
profiler = RealTimeProfiler()

landmarker = FaceLandmark()
aligner = FaceAligner()
tvl1 = TVL1(fast_mode=True)

roi_defs = FaceRoiPoints.ALL_ROIS
tile_size = (32, 32)
margin = 0.05
cols = 3
rows = int(np.ceil(len(roi_defs) / cols))

sequences: list[FrameSequence] = []
clipped_sequences: list[FrameSequence] = []
all_features = []
labels = []
groups = []
spotted_intervals = []
gt_intervals = []

print("Extracting ROI optical flow for SAMM video clips...")


Loaded SAMM metadata with 159 total entries.
Extracting ROI optical flow for SAMM video clips...


W0000 00:00:1788746077.407149  172257 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1788746077.412549  172733 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788746077.434076  172738 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788746077.438303  172257 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
W0000 00:00:1788746077.444220  172770 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788746077.460272  172775 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inferen

## 2. Extract ROI flows & spot micro-expression boundaries

In [3]:
for _, row in df.iterrows():
    subject = row['Subject']
    subject_str = f"{int(subject):03d}"
    filename = str(row['Filename'])
    onset_f = int(row['Onset Frame'])
    offset_f = int(row['Offset Frame'])
    raw_emotion = row['Estimated Emotion']

    label = MELabel.map(raw_emotion, target='2-class')
    if label is None:
        continue

    video_dir = os.path.join(samm_dir, subject_str, filename)
    img_paths = sorted(glob.glob(os.path.join(video_dir, "*.jpg")))

    sliced_paths = []
    for path in img_paths:
        fname = os.path.basename(path)
        frame_num = int(fname.split('_')[-1].split('.')[0])
        if onset_f <= frame_num <= offset_f:
            sliced_paths.append(path)

    if len(sliced_paths) < 2:
        continue

    crops_list = []
    for path in sliced_paths:
        frame = cv2.imread(path)
        if frame is None:
            continue
        landmarks = landmarker.detect(frame)
        try:
            _, aligned_landmarks = aligner.align_with_landmarks(image=frame, landmarks=landmarks)
        except Exception:
            aligned_landmarks = landmarks

        crops = []
        for roi_points in roi_defs:
            try:
                roi, _ = landmarker.crop_roi(
                    image=frame,
                    landmark_result=aligned_landmarks,
                    roi_points=roi_points,
                    margin=margin,
                    target_size=tile_size,
                )
            except Exception:
                roi = np.zeros((tile_size[1], tile_size[0], 3), dtype=np.uint8)
            crops.append(roi)
        crops_list.append(crops)

    if len(crops_list) < 2:
        continue

    flows_per_roi = []
    for i in range(len(crops_list) - 1):
        pairs = list(zip(crops_list[i], crops_list[i + 1]))
        flows_batch = tvl1.compute_batch(pairs, download=True)
        flows_per_roi.append(np.stack(flows_batch, axis=0)) # (5, H, W, 2)

    # (T, 5, 2, H, W)
    flow_arr = np.stack(flows_per_roi, axis=0).transpose(0, 1, 4, 2, 3)
    seq = FrameSequence(subject=subject_str, clip_name=filename, flow=flow_arr, label=label, fps=FPS)

    gt_interval = (0, len(seq) - 1)

    # Best-IoU match against gt (standard detection-eval assignment), not
    # spot()'s arbitrary first-detected-candidate selection.
    candidates = spotter.spot_all(seq.magnitudes)
    match_result = spotter.match(candidates, gt_interval, fallback_mags=seq.magnitudes)
    feat_interval = match_result.get('feat_interval', gt_interval)
    spot_interval = match_result.get('spot_interval', gt_interval)

    clipped_seq = seq.clip(feat_interval[0], feat_interval[1])
    if len(clipped_seq) == 0:
        continue

    sequences.append(seq)
    clipped_sequences.append(clipped_seq)
    spotted_intervals.append(spot_interval)
    gt_intervals.append(gt_interval)
    labels.append(label)
    groups.append(subject_str)

    flow_tensor = torch.from_numpy(clipped_seq.flow)
    feats = extractor._extract(flow_tensor).cpu().numpy()
    all_features.append(feats)

print(f"Extracted {len(sequences)} SAMM sequences.")

W0000 00:00:1788746077.498428  172766 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
/home/inadio/skripkir/pulse-live/.venv/lib/python3.12/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Extracted 118 SAMM sequences.


## 3. Spotting Performance Benchmark (Fang et al. 2023 protocol)

In [4]:
spot_metrics = spotter.benchmark(spotted_intervals, gt_intervals, iou_thresh=0.5)
print("\n=== Spotting Performance (SAMM | Fang et al. 2023 Standard) ===")
print(f"Total Samples:       {spot_metrics['samples']}")
print(f"True Positives:      {spot_metrics['tp']} (IoU >= 0.5)")
print(f"Average IoU:         {spot_metrics['mean_iou']:.4f}")
print(f"Spotting Precision:  {spot_metrics['precision']:.4f}")
print(f"Spotting Recall:     {spot_metrics['recall']:.4f}")
print(f"Spotting F1-Score:   {spot_metrics['f1']:.4f}")
print("===============================================================\n")


=== Spotting Performance (SAMM | Fang et al. 2023 Standard) ===
Total Samples:       118
True Positives:      118 (IoU >= 0.5)
Average IoU:         0.8967
Spotting Precision:  1.0000
Spotting Recall:     1.0000
Spotting F1-Score:   1.0000



## 4. Feature Matrix & LOSO Cross-Validation (Baseline SVM)

In [5]:
y, class_names = MELabel.encode(labels)
groups_arr = np.array(groups)
X_static = np.stack([np.concatenate([f.mean(axis=0), f.std(axis=0)]) for f in all_features])

logo = LeaveOneGroupOut()
splits = list(logo.split(X_static, y, groups=groups_arr))
print(f"LOSO Cross-Validation | Total Subjects (Splits): {len(splits)}")

svm_preds = np.zeros_like(y)
for train_idx, test_idx in splits:
    X_train, y_train = X_static[train_idx], y[train_idx]
    X_test, y_test = X_static[test_idx], y[test_idx]

    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    clf = SVC(kernel='rbf', C=2.0, class_weight='balanced', random_state=42)
    clf.fit(X_train_scaled, y_train)
    svm_preds[test_idx] = clf.predict(X_test_scaled)

print("\n=== SAMM SVM Classification Performance (LOSO) ===")
print(f"Accuracy:        {accuracy_score(y, svm_preds):.4f}")
print(f"Macro F1-Score:  {f1_score(y, svm_preds, average='macro'):.4f}")
print(f"Macro Precision: {precision_score(y, svm_preds, average='macro', zero_division=0):.4f}")
print(f"Macro Recall:    {recall_score(y, svm_preds, average='macro', zero_division=0):.4f}")
print("===================================================")

LOSO Cross-Validation | Total Subjects (Splits): 28

=== SAMM SVM Classification Performance (LOSO) ===
Accuracy:        0.7627
Macro F1-Score:  0.5189
Macro Precision: 0.5830
Macro Recall:    0.5305


## 5. SpatioTemporalCNN 3D Deep Learning Classification (LOSO)

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nEvaluating SpatioTemporalCNN 3D Deep Learning model on device: {device}...")

cnn_preds = np.zeros_like(y)
batch_size = 8

for fold_idx, (train_idx, test_idx) in enumerate(splits):
    train_seqs = [clipped_sequences[i] for i in train_idx]
    test_seqs = [clipped_sequences[i] for i in test_idx]
    y_train = torch.tensor(y[train_idx], dtype=torch.long, device=device)

    model = SpatioTemporalCNN(in_channels=10, num_classes=len(class_names), dropout_p=0.3).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(10):
        perm = torch.randperm(len(train_seqs))
        for i in range(0, len(train_seqs), batch_size):
            indices = perm[i : i + batch_size]
            if len(indices) < 2:
                continue
            batch_seqs = [train_seqs[idx] for idx in indices]
            batch_x = FrameSequence.pad_batch(batch_seqs, max_len=64, device=device)
            batch_y = y_train[indices]

            optimizer.zero_grad()
            out = model(batch_x)
            loss = criterion(out, batch_y)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        test_fold_preds = []
        for i in range(0, len(test_seqs), batch_size):
            batch_seqs = test_seqs[i : i + batch_size]
            batch_x = FrameSequence.pad_batch(batch_seqs, max_len=64, device=device)
            out_test = model(batch_x)
            test_fold_preds.append(torch.argmax(out_test, dim=1).cpu().numpy())
        if test_fold_preds:
            cnn_preds[test_idx] = np.concatenate(test_fold_preds)

print("\n=== SAMM SpatioTemporalCNN Classification Performance (LOSO) ===")
print(f"Accuracy:        {accuracy_score(y, cnn_preds):.4f}")
print(f"Macro F1-Score:  {f1_score(y, cnn_preds, average='macro'):.4f}")
print(f"Macro Precision: {precision_score(y, cnn_preds, average='macro', zero_division=0):.4f}")
print(f"Macro Recall:    {recall_score(y, cnn_preds, average='macro', zero_division=0):.4f}")
print("=================================================================")
print("Classification Report:")
print(classification_report(y, cnn_preds, target_names=class_names, zero_division=0))


Evaluating SpatioTemporalCNN 3D Deep Learning model on device: cuda...

=== SAMM SpatioTemporalCNN Classification Performance (LOSO) ===
Accuracy:        0.6949
Macro F1-Score:  0.4972
Macro Precision: 0.5011
Macro Recall:    0.5008
Classification Report:
              precision    recall  f1-score   support

    negative       0.78      0.85      0.81        92
    positive       0.22      0.15      0.18        26

    accuracy                           0.69       118
   macro avg       0.50      0.50      0.50       118
weighted avg       0.66      0.69      0.67       118



## 6. Real-Time End-to-End Latency Benchmarking with RealTimeProfiler

In [7]:
print("\n=== Running Real-Time Performance Profiling ===")
cnn_full = SpatioTemporalCNN(in_channels=10, num_classes=len(class_names)).to(device)
cnn_full.eval()

frame_counts = [len(s) for s in sequences]

for seq in sequences:
    with profiler.record("spot"):
        feat_int, _ = spotter.spot(seq.magnitudes, fallback_half_win=25)

    clipped = seq.clip(feat_int[0], feat_int[1])
    if len(clipped) == 0:
        continue

    with profiler.record("cnn_infer"):
        x_cnn = clipped.to_tensor(device=device)
        with torch.no_grad():
            _ = cnn_full(x_cnn)

print("\nReal-Time Performance Statistics:")
print(profiler.report(frame_counts).to_string(index=False))


=== Running Real-Time Performance Profiling ===

Real-Time Performance Statistics:
                 Metric            Value
          Latency: Spot   1.188 ms / seq
     Latency: Cnn_infer   3.804 ms / seq
 Total Sequence Latency         4.992 ms
Average Sequence Length      72.1 frames
Estimated Frame Latency 0.069 ms / frame
       Throughput (FPS)      14443.9 FPS
